# Tomato Disease CNN Training (Google Colab)

This notebook trains a TensorFlow CNN for 6 tomato leaf classes and saves the model to Google Drive.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Set Paths

Put your dataset in Drive with this structure:
- Tomato/Train/<class folders>
- Tomato/Val/<class folders>
- Tomato/Test/<class folders>

In [ ]:
from pathlib import Path
import json

DATASET_ROOT = Path('/content/drive/MyDrive/Tomato')
OUTPUT_ROOT = Path('/content/drive/MyDrive/Real_Project_Outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Dataset root:', DATASET_ROOT)
print('Output root:', OUTPUT_ROOT)

In [ ]:
CLASS_NAMES = [
    'Bacterial Spot',
    'Early Blight',
    'Late Blight',
    'Healthy',
    'Septoria Leaf Spot',
    'Yellow Leaf Curl Virus',
]
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def resolve_split_dir(root: Path, split_name: str) -> Path:
    candidates = [split_name.lower(), split_name.capitalize(), split_name.upper()]
    for c in candidates:
        p = root / c
        if p.exists():
            return p
    raise FileNotFoundError(f'Missing split {split_name}. Tried {candidates}')

def count_images(folder: Path) -> int:
    return sum(1 for f in folder.iterdir() if f.is_file() and f.suffix.lower() in EXTS)

stats = {}
for split in ['Train', 'Val', 'Test']:
    sp = resolve_split_dir(DATASET_ROOT, split)
    stats[split] = {name: count_images(sp / name) for name in CLASS_NAMES}

print(json.dumps(stats, indent=2))

In [ ]:
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10

train_dir = resolve_split_dir(DATASET_ROOT, 'train')
val_dir = resolve_split_dir(DATASET_ROOT, 'val')
test_dir = resolve_split_dir(DATASET_ROOT, 'test')

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='int',
    class_names=CLASS_NAMES,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    labels='inferred',
    label_mode='int',
    class_names=CLASS_NAMES,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels='inferred',
    label_mode='int',
    class_names=CLASS_NAMES,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

In [ ]:
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.12),
])

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    data_aug,
    tf.keras.layers.Rescaling(1.0 / 255),
    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(256, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

model.summary()

In [ ]:
best_model_path = OUTPUT_ROOT / 'tomato_disease_cnn.h5'
history_path = OUTPUT_ROOT / 'training_history.json'
metrics_path = OUTPUT_ROOT / 'test_metrics.json'

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(str(best_model_path), monitor='val_accuracy', save_best_only=True, verbose=1),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

hist = {k: [float(x) for x in v] for k, v in history.history.items()}
history_path.write_text(json.dumps(hist, indent=2), encoding='utf-8')
print('History saved to:', history_path)

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=1)
metrics = {
    'test_loss': float(test_loss),
    'test_accuracy': float(test_acc),
    'class_names': CLASS_NAMES
}
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
model.save(str(best_model_path))
print(json.dumps(metrics, indent=2))
print('Model saved to:', best_model_path)